In [ ]:
# saving the model and preprocessing pipeline using joblib, and reuse the model later 
# for inference on new data (input.csv). This approach helps avoid retraining the model
# every time, improving performance and enabling reproducibility.

# Saving the model (model.pkl) and preprocessing pipeline (pipeline.pkl) ensures we
# can quickly load and run inference anytime in the future.

# Why we used Joblib:
# joblib efficiently serializes large NumPy arrays (like in sklearn models).
# Faster and more suitable than pickle for scikit-learn objects.

# Why the If-Else Logic:
# The program checks if a saved model exists.
# If not, it trains and saves the model.
# If it does, it skips training and only runs inference, saving time.

In [25]:
import os
import joblib
import pandas as pd 
import numpy as np 
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor

In [29]:
MODEL_FILE = "model.pkl"
PIPELINE_FILE = "pipeline.pkl"

In [30]:
def build_pipeline(num_att, cat_att): 

    cat_pipe = Pipeline([
        ("OneHot", OneHotEncoder(handle_unknown="ignore"))
    ])
    
    num_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("stand", StandardScaler())
    ])

    full_pipe = ColumnTransformer([
        ("cat", cat_pipe, cat_att),
        ("num", num_pipe, num_att) 
        
    ])
    return full_pipe
    

In [31]:
if not os.path.exists(MODEL_FILE): 
    # training 
    df = pd.read_csv("housing.csv")

    df['income_cat'] = pd.cut(df['median_income'], bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf], labels=[1,2,3,4,5])
    strat = StratifiedShuffleSplit(n_splits=1, random_state=42, test_size=0.2)
    for train_idx, test_idx in strat.split(df, df['income_cat']):
        train_data = df.loc[train_idx].drop('income_cat', axis=1)
        df.loc[test_idx].drop('income_cat', axis=1).to_csv("input.csv", index=False) # saving test set as separate file

    x = train_data.drop('median_house_value', axis=1) 
    y = train_data['median_house_value']

    num_att = x.drop('ocean_proximity', axis=1).columns.to_list()
    cat_att = ['ocean_proximity']

    pipeline = build_pipeline(num_att, cat_att)
    x_transformed = pipeline.fit_transform(x)

    model = RandomForestRegressor(random_state=42)
    model.fit(x_transformed,y)

    # save model and pipeline 
    joblib.dump(model, MODEL_FILE)
    joblib.dump(pipeline, PIPELINE_FILE)

    print("Model trained and saved successfully.")

else:
    model = joblib.load(MODEL_FILE)
    pipeline = joblib.load(PIPELINE_FILE)

    input_data = pd.read_csv("input.csv")
    transformed_inp = pipeline.transform(input_data)
    model_pred = model.predict(transformed_inp)

    actual = input_data['median_house_value'].copy()

    input_data['median_house_value'] = model_pred
    in

    input_data.to_csv("output.csv", index=False)
    print("model predicts successfully! check out output.csv file.")
    

model predicts successfully! check out output.csv file.
